# The intersect3 problem

## Statement
> Compute the intersection of **three** sets of (distinguished) positive integers. 

> For a typical problem, each of the three sets contains less than 1000 integers. All the integers are less than $10^8$. 

## Background
> The problem comes from the implementation of k-nearest-neighbor algorithm for a set of three-dimensional mesh points of size N. Such algorithm is a key component in the 3D mesh generation algorithm for finite element analysis. Naive implementations are usually very slow, because to find the k-nearest neighbors for a mesh point, the  Euclidean distances to all the N-1 neighbors are calculated. 

> The simplest improvement is to draw a cube at one mesh point and select all the mesh points within the cube. If the density of mesh points are uniform, then the number of points inside the cube would be enough for the k-nearest neighbors. The Euclidean distance computation and sorting can be limited within this cube to speed up the k-nearest neighbor search. For a 3D mesh of $N$ points,  drawing a cube would reduce the number of distance calculations by a factor of $O(N/m)$ where $m$ is the number of mesh point inside the cube. That is a great improvement because the cube typically contains no more than 1000 mesh points.          

________________

## Convenient implementation
> Use the build-in Julia funcition `intersect`. This function is actually [not trivial at all](https://github.com/JuliaLang/julia/issues/13675). A quick look at the implementation of this function in the file `array.jl:2625` and `abstractset.jl:128` (Julia Version 1.7.2 (2022-02-06)), I found that this function rely on the `Set` data structure. 

In [1]:
intersect3_v1(A,B,C) = intersect(intersect(A,B),C)

intersect3_v1 (generic function with 1 method)

> There are discussions 
> [here](https://discourse.julialang.org/t/intersect-is-slow-for-large-arrays/442/3)
> and 
> [here](https://github.com/JuliaLang/julia/issues/13675) 
> pointing out the performance bug of this build-in function. 
> Based on the discussions there, and the source code mentioned earlier, 
> I sort the length of three sets and put the smallest set to the first argument. 
> Benchmark results show that it indeed improves the performance. 

In [2]:
function intersect3_v2(A,B,C)
    lenA,lenB,lenC = length(A),length(B),length(C)
    if lenA <= lenB
        if lenB <= lenC
            return intersect(intersect(A,B),C)
        else
            return (lenA <= lenC ? intersect(intersect(A,C),B) 
                                 : intersect(intersect(C,A),B))
        end
    else # lenB < lenA
        if lenA <= lenC
            return intersect(intersect(B,A),C)
        else
            return (lenB <= lenC ? intersect(intersect(B,C),A)
                                 : intersect(intersect(C,B),A))
        end
    end
end

intersect3_v2 (generic function with 1 method)

In [3]:
using BenchmarkTools

In [5]:
b1 = @benchmarkable intersect3_v1(A,B,C)  setup=(
    A = unique(rand(1:3000000,rand(100:90000))); 
    B = unique(rand(1:3000000,rand(100:90000)));
    C = unique(rand(1:3000000,rand(100:90000))))
tune!(b1)
run(b1)

BenchmarkTools.Trial: 446 samples with 1 evaluation.
 Range (min … max):  575.506 μs … 11.698 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):       4.802 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):     5.057 ms ±  2.286 ms  ┊ GC (mean ± σ):  0.63% ± 2.43%

         ▆▁▂▂▅▃▃█▅▁▁▆▇██▂▃▄ ▃▇▃  ▂▃▂ ▆▁▁▃▁▄▁▂▃   ▁              
  ▄▃▁▄▄▃▇██████████████████▇████▇███▇███████████▇██▇▅▇▅▅▄▆▄▅▅▃ ▆
  576 μs          Histogram: frequency by time         10.2 ms <

 Memory estimate: 222.41 KiB, allocs estimate: 35.

In [6]:
b2 = @benchmarkable intersect3_v2(A,B,C)  setup=(
    A = unique(rand(1:3000000,rand(100:90000))); 
    B = unique(rand(1:3000000,rand(100:90000)));
    C = unique(rand(1:3000000,rand(100:90000))))
tune!(b2)
run(b2)

BenchmarkTools.Trial: 490 samples with 1 evaluation.
 Range (min … max):  323.362 μs … 9.285 ms  ┊ GC (min … max): 0.00% … 4.49%
 Time  (median):       3.448 ms             ┊ GC (median):    0.00%
 Time  (mean ± σ):     3.844 ms ± 1.838 ms  ┊ GC (mean ± σ):  1.20% ± 4.09%

           ▅▁█▅▆▆▄▃▄▁▇▅▇▁ ▄▁   ▅▁▁ ▂▁ ▂▃▂  ▁                   
  ▃▃▄▃▄▅▇▅▆██████████████▅██▇█████▆██▄███▇▃██▃▆▅▅▆▃▄▅▄▃▃▃▁▃▁▄ ▅
  323 μs          Histogram: frequency by time        8.64 ms <

 Memory estimate: 267.95 KiB, allocs estimate: 33.

> Later in this note, the benchmark results tell us that this is the fastest implementation.

________________

## Counting, when maximum of A,B,C is known
> When we are guaranteed that `A,B,C` contain integers no greater than `N`, we may simply count the number of occurances of each element in these sets and select the elements that occur exactly three times. For small values of `N`, the following simple counting method may be faster.
> The following implementation `intersect3_v3` is simple enough but the complexity is $O(N)$. 
> When the three sets `A,B,C` is small and sparse in the range `1:N`, 
> the perfornamce is far from optimal.  

In [4]:
function intersect3_v3(A,B,C; N=1_000_000)
    cnt = zeros(Int,N)
    cnt[A] .+= 1
    cnt[B] .+= 1
    cnt[C] .+= 1
    return findall(cnt.==3)
end

intersect3_v3 (generic function with 1 method)

> The benchmark is done for large and small values of `N`. For large value `N=3000000`, I find

In [7]:
b3 = @benchmarkable intersect3_v3(A,B,C;N=3000000)  setup=(
    A = unique(rand(1:3000000,rand(100:90000))); 
    B = unique(rand(1:3000000,rand(100:90000)));
    C = unique(rand(1:3000000,rand(100:90000))))
tune!(b3)
run(b3)

BenchmarkTools.Trial: 272 samples with 1 evaluation.
 Range (min … max):   4.390 ms … 64.715 ms  ┊ GC (min … max): 0.00% … 80.18%
 Time  (median):     11.817 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   11.359 ms ±  4.101 ms  ┊ GC (mean ± σ):  4.44% ±  8.53%

                                   ▁▅▆▇█▇▆▁▁                   
  ▃▃▃▄▁▅▅▅▃▄▃▄▆▃▄▃▅▅▃▁▁▁▃▁▁▃▁▃▃▅████████████▇▇▅▃▅▁▁▃▄▃▃▃▃▁▃▁▃ ▃
  4.39 ms         Histogram: frequency by time        16.5 ms <

 Memory estimate: 23.47 MiB, allocs estimate: 11.

> For small value `N=10000`, the result for `intersect3_v3` is much better:

In [8]:
b2 = @benchmarkable intersect3_v2(A,B,C)  setup=(
    A = unique(rand(1:10000,rand(100:9000))); 
    B = unique(rand(1:10000,rand(100:9000)));
    C = unique(rand(1:10000,rand(100:9000))))
tune!(b2)
run(b2)

BenchmarkTools.Trial: 5549 samples with 1 evaluation.
 Range (min … max):   33.235 μs …   1.753 ms  ┊ GC (min … max): 0.00% … 48.10%
 Time  (median):     309.677 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   343.754 μs ± 187.047 μs  ┊ GC (mean ± σ):  0.86% ±  4.38%

        ▄▆▆▇█▇▇▇█▆▅▅▄▄▄▅▃▃▄▄ ▂▂▂▂▂▁                              
  ▂▂▃▇▇████████████████████████████▇█▆▇▆▅▅▅▅▅▄▄▄▄▄▄▃▄▃▂▂▂▂▂▂▂▂▁ ▅
  33.2 μs          Histogram: frequency by time          874 μs <

 Memory estimate: 35.43 KiB, allocs estimate: 33.

In [9]:
b3 = @benchmarkable intersect3_v3(A,B,C;N=10000)  setup=(
    A = unique(rand(1:10000,rand(100:9000))); 
    B = unique(rand(1:10000,rand(100:9000)));
    C = unique(rand(1:10000,rand(100:9000))))
tune!(b3)
run(b3)

BenchmarkTools.Trial: 8408 samples with 1 evaluation.
 Range (min … max):  14.543 μs …  1.310 ms  ┊ GC (min … max): 0.00% … 88.98%
 Time  (median):     42.867 μs              ┊ GC (median):    0.00%
 Time  (mean ± σ):   49.466 μs ± 61.168 μs  ┊ GC (mean ± σ):  8.24% ±  6.56%

            ▁▃▄▆▇▇▆██▆▄▃▂                                      
  ▁▁▂▂▃▃▄▆███████████████▇▆▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁ ▃
  14.5 μs         Histogram: frequency by time         112 μs <

 Memory estimate: 92.75 KiB, allocs estimate: 9.

________________

## Exploit the orderedness of integer set
> The integer sets can be sorted. With sorted lists $A,B,C$, 
> the intersection algorithm can be faster because,
> when we build the set $I = A\cap B\cap C$, 
> each time we reject an element, say $a \in A$, from being added into $I$, 
> we can jump in the the other two sorted lists to skip the elements 
> which are smaller than $a$. 

> The following version is a fakakta implementation based on this idea. 

In [10]:
function bisec(a::Int, b::Int, x::Int, Lref::Base.RefValue{Vector{Int64}})::Int
    (x <  Lref[][a]) && (return a-1)
    (x >= Lref[][b]) && (return b)
    c = 0
    while b - a > 1
        c = (a + b) ÷ 2
        if x < Lref[][c]
            b = c
        else
            a = c
        end
    end
    x < Lref[][b] ? a : b
end

bisec (generic function with 1 method)

In [11]:
function ord3!(ord,s,t)
    if s[1]-t[1] <= s[2]-t[2]
        return (s[2]-t[2] <= s[3]-t[3] 
                    ? (ord[1]=1; ord[2]=2; ord[3]=3;)
                    : (s[1]-t[1] <= s[3]-t[3] 
                            ? (ord[1]=1;ord[2]=3;ord[3]=2;) 
                            : (ord[1]=3;ord[2]=1;ord[3]=2;)))
    else # s[2] < s[1]
        return (s[1]-t[1] <= s[3]-t[3] 
                    ? (ord[1]=2;ord[2]=1;ord[3]=3;)
                    : (s[2]-t[2] <= s[3]-t[3] 
                            ? (ord[1]=2;ord[2]=3;ord[3]=1;) 
                            : (ord[1]=3;ord[2]=2;ord[3]=1;)))
    end    
end

allposv(v) = v[1]>0 && v[2]>0 && v[3]>0

function intersect3_v4(A,B,C)
    # profiling shows that the major efforts are sorting
    As = sort(A)
    Bs = sort(B)
    Cs = sort(C)
    # use reference to avoid allocations
    Lref = [Ref(As),Ref(Bs),Ref(Cs)] ;
    # preparations
    maxt = maximum(r->first(r[]), Lref)
    mint = minimum(r->last(r[]), Lref)
    head = [bisec(1,length(r[]),maxt,r) for r in Lref]
    tail = [bisec(1,length(r[]),mint,r) for r in Lref]
    s = [-1,-1,-1]  #+ reduce alloc
    ord3!(s, tail, head)  #+ reduce alloc
    if !(allposv(head) && head[s[1]]<=tail[s[1]])
        return []
    end
    intsct = zeros(Int,minimum(tail.-head))
    p = 1
    while allposv(head) && head[s[1]]<=tail[s[1]]
        ord3!(s, tail, head)  #+ reduce alloc
        a = Lref[s[1]][][head[s[1]]]
        head[s[1]] = head[s[1]]+1
        # search in L[head:tail]
        i2 = bisec(head[s[2]], tail[s[2]], a, Lref[s[2]])
        if i2 < head[s[2]]
            # outside range head:tail
            continue
        elseif a != Lref[s[2]][][i2]
            head[s[2]] = i2+1
            continue
        else # a == Lref[s[2]][][i2]
            i3 = bisec(head[s[3]], tail[s[3]], a, Lref[s[3]])
            if i3 < head[s[3]]
                head[s[2]] = i2+1
                continue
            else
                if a == Lref[s[3]][][i3]
                    intsct[p] = a
                    p += 1
                end
                head[s[2]] = i2+1
                head[s[3]] = i3+1
                continue
            end
        end
    end
    intsct[1:(p-1)]
end


intersect3_v4 (generic function with 1 method)

> Of course, fakakta does not imply broken. This implementation is at least correct:

In [12]:
for i=1:10000
    A = unique(rand(1:3000,rand(300:5000)))
    B = unique(rand(1:1000,rand(300:5000)))
    C = unique(rand(1:6000,rand(300:5000)))
    @assert sort(intersect3_v2(A,B,C))==intersect3_v4(A,B,C) 
end

> Now we do benchmark to compare `intersect3_v4` against `intersect3_v2`:

> Small size:

In [13]:
b2 = @benchmarkable intersect3_v2(A,B,C)  setup=(
    A = unique(rand(1:10000,rand(100:9000))); 
    B = unique(rand(1:10000,rand(100:9000)));
    C = unique(rand(1:10000,rand(100:9000))))
tune!(b2)
run(b2)

BenchmarkTools.Trial: 5600 samples with 1 evaluation.
 Range (min … max):   22.710 μs …   1.563 ms  ┊ GC (min … max): 0.00% … 57.64%
 Time  (median):     304.881 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   341.994 μs ± 187.076 μs  ┊ GC (mean ± σ):  1.17% ±  4.87%

        ▂▃▆█▇▇▆▆█▆▅▅▅▇▄▄▂▂▃▂▂▁▃▃▂                                
  ▁▂▃▄▆█████████████████████████████▇▇▆▆▆▅▅▅▅▄▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▂▁ ▅
  22.7 μs          Histogram: frequency by time          863 μs <

 Memory estimate: 23.40 KiB, allocs estimate: 33.

In [14]:
b4 = @benchmarkable intersect3_v4(A,B,C)  setup=(
    A = unique(rand(1:10000,rand(100:9000))); 
    B = unique(rand(1:10000,rand(100:9000)));
    C = unique(rand(1:10000,rand(100:9000))))
tune!(b4)
run(b4)

BenchmarkTools.Trial: 4493 samples with 1 evaluation.
 Range (min … max):   62.003 μs …   1.842 ms  ┊ GC (min … max): 0.00% … 50.40%
 Time  (median):     541.109 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   560.812 μs ± 214.898 μs  ┊ GC (mean ± σ):  0.26% ±  2.46%

              ▁▁▄▄▃▅▆▆▆███▇▇█▇▇▆▇▆▇▄▃▄▅▄▃ ▂▁ ▁                   
  ▂▂▂▂▃▂▃▄▅▅▇▇██████████████████████████████▇██▇▆▆▆▄▆▄▄▃▄▃▂▃▂▂▂ ▆
  62 μs            Histogram: frequency by time          1.1 ms <

 Memory estimate: 15.23 KiB, allocs estimate: 13.

> Large size:

In [15]:
b2 = @benchmarkable intersect3_v2(A,B,C)  setup=(
    A = unique(rand(1:3000000,rand(100:90000))); 
    B = unique(rand(1:3000000,rand(100:90000)));
    C = unique(rand(1:3000000,rand(100:90000))))
tune!(b2)
run(b2)

BenchmarkTools.Trial: 522 samples with 1 evaluation.
 Range (min … max):  532.447 μs … 10.047 ms  ┊ GC (min … max): 0.00% … 4.49%
 Time  (median):       3.258 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):     3.561 ms ±  1.854 ms  ┊ GC (mean ± σ):  1.07% ± 3.49%

        ▂▁▄▄▆▃▅ ▂█▄▃▅▃▅▂ ▃ ▃▂                                   
  ▄▅▆▇█████████▇██████████████▇▇▄▇▆▇▅▆▆█▄▇▇▄▃▆▃▄▃▃▃▃▄▄▄▄▃▃▁▁▁▄ ▅
  532 μs          Histogram: frequency by time         8.97 ms <

 Memory estimate: 318.80 KiB, allocs estimate: 33.

In [16]:
b4 = @benchmarkable intersect3_v4(A,B,C)  setup=(
    A = unique(rand(1:3000000,rand(100:90000))); 
    B = unique(rand(1:3000000,rand(100:90000)));
    C = unique(rand(1:3000000,rand(100:90000))))
tune!(b4)
run(b4)

BenchmarkTools.Trial: 351 samples with 1 evaluation.
 Range (min … max):  1.343 ms … 17.190 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     8.104 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   8.263 ms ±  3.003 ms  ┊ GC (mean ± σ):  0.23% ± 1.20%

              ▁  ▃  ▂▁█▇▅▇█▃ ▁▃▂▂▅▄▄ █▄▁▅    ▁                
  ▄▃▁▅▄▁█▄▅█▄██▅██▆▇█████████████████████▅█▃▇█▅▄█▅█▁▅▆▁▁▃▃▃▃ ▅
  1.34 ms        Histogram: frequency by time        15.6 ms <

 Memory estimate: 241.84 KiB, allocs estimate: 15.

> The following implementation `` assumes that the input lists are sorted:

In [18]:
# for sorted integer lists
function intersect3_v4s(As,Bs,Cs)
    # use reference to avoid allocations
    Lref = [Ref(As),Ref(Bs),Ref(Cs)] ;
    # preparations
    maxt = maximum(r->first(r[]), Lref)
    mint = minimum(r->last(r[]), Lref)
    head = [bisec(1,length(r[]),maxt,r) for r in Lref]
    tail = [bisec(1,length(r[]),mint,r) for r in Lref]
    s = [-1,-1,-1]  #+ reduce alloc
    ord3!(s, tail, head)  #+ reduce alloc
    if !(allposv(head) && head[s[1]]<=tail[s[1]])
        return []
    end
    intsct = zeros(Int,minimum(tail.-head))
    p = 1
    while allposv(head) && head[s[1]]<=tail[s[1]]
        ord3!(s, tail, head)  #+ reduce alloc
        a = Lref[s[1]][][head[s[1]]]
        head[s[1]] = head[s[1]]+1
        # search in L[head:tail]
        i2 = bisec(head[s[2]], tail[s[2]], a, Lref[s[2]])
        if i2 < head[s[2]]
            # outside range head:tail
            continue
        elseif a != Lref[s[2]][][i2]
            head[s[2]] = i2+1
            continue
        else # a == Lref[s[2]][][i2]
            i3 = bisec(head[s[3]], tail[s[3]], a, Lref[s[3]])
            if i3 < head[s[3]]
                head[s[2]] = i2+1
                continue
            else
                if a == Lref[s[3]][][i3]
                    intsct[p] = a
                    p += 1
                end
                head[s[2]] = i2+1
                head[s[3]] = i3+1
                continue
            end
        end
    end
    intsct[1:(p-1)]
end


intersect3_v4s (generic function with 1 method)

> You can try out the benchmark yourself to see the performance improvement of `intersect3_v4s`.
> To be fair, we use the sorted lists for all benchmark cases.

In [22]:
b4s = @benchmarkable intersect3_v4s(As,Bs,Cs)  setup=(
    As = sort(unique(rand(1:3000000,rand(100:90000)))); 
    Bs = sort(unique(rand(1:3000000,rand(100:90000))));
    Cs = sort(unique(rand(1:3000000,rand(100:90000)))))
tune!(b4s)
run(b4s)

BenchmarkTools.Trial: 340 samples with 1 evaluation.
 Range (min … max):   27.292 μs …   3.058 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     960.100 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):     1.064 ms ± 713.242 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █ ▇▂ ▂▅▅▂▅▂▆▇▅▅▃▆▅ ▄ ▇▄ ▄ ▆ ▃  ▂      ▂                        
  █▆████████████████▅████▇█▇█▅██████▄▇▆▄██▄▅▅▄▅▄▃▃▃▃▃▃▃▃▇▃▃▁▃▃▅ ▅
  27.3 μs          Histogram: frequency by time         2.98 ms <

 Memory estimate: 1.83 KiB, allocs estimate: 10.

In [ ]:
b4 = @benchmarkable intersect3_v4(As,Bs,Cs)  setup=(
    As = sort(unique(rand(1:3000000,rand(100:90000)))); 
    Bs = sort(unique(rand(1:3000000,rand(100:90000))));
    Cs = sort(unique(rand(1:3000000,rand(100:90000)))))
tune!(b4)
run(b4)

BenchmarkTools.Trial: 310 samples with 1 evaluation.
 Range (min … max):  300.952 μs … 5.499 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):       1.838 ms             ┊ GC (median):    0.00%
 Time  (mean ± σ):     2.048 ms ± 1.087 ms  ┊ GC (mean ± σ):  0.41% ± 2.71%

       ▁   ▅█▁▇  ▄ ▁ ▁ ▁▃    ▁▁                                
  ▄▃▇▄▃█▆▆▇█████▅█▆██████▆▆▇▄██▄▄▅▄▆▆▄▆▇▄▅▃▅▁▄▃▃▃▃▁▃▃▄▃▃▃▃▃▁▃ ▄
  301 μs          Histogram: frequency by time        4.97 ms <

 Memory estimate: 246.44 KiB, allocs estimate: 14.

In [24]:
b2 = @benchmarkable intersect3_v2(As,Bs,Cs)  setup=(
    As = sort(unique(rand(1:3000000,rand(100:90000)))); 
    Bs = sort(unique(rand(1:3000000,rand(100:90000))));
    Cs = sort(unique(rand(1:3000000,rand(100:90000)))))
tune!(b2)
run(b2)

BenchmarkTools.Trial: 271 samples with 1 evaluation.
 Range (min … max):  421.308 μs … 9.903 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):       3.625 ms             ┊ GC (median):    0.00%
 Time  (mean ± σ):     3.943 ms ± 1.971 ms  ┊ GC (mean ± σ):  0.90% ± 2.82%

      ▁▁    ▃▃▂▂▂▁▅ ▃▅▆▄ ▃█    ▂     ▁                         
  ▃▁▃▆██▇█▆█████████████▇██▆▇▇▆█▇▆█▅▅█▄▅▅▄▅▆▁▇▁▅▃▃▄▄▄▃▁▃▁▁▁▃▃ ▄
  421 μs          Histogram: frequency by time        9.58 ms <

 Memory estimate: 319.45 KiB, allocs estimate: 33.

## Conclusion
> The best algorithm for intersect3 problem depends on the scales:
> 1. for sets with elements less than $N \sim 10^4$, a direct counting is simple and fast;
> 2. for sets with elements less than $N \sim 10^8$, the Julia implementation wins;
> 3. if the sets are given as sorted lists, the fakakta implementation `intersect3_v4s` is the best.